# Sistema de Teleoperación y Visión 3D para Mano Robótica Inspire RH56DFTP-2R
Sistema de teleoperación controlado por Intel RealSense Depth Camera para la mano robótica Inspire RH56DFTP-2R, es una interface de control de operación para la mano de 6 grado de libertas (DOF), adicional, se encuentra descrito de forma exhaustiva, bloque por bloque, la arquitectura, las ecuaciones cinemáticas, la máquina de estados y la comunicación por registros Modbus TCP del script de teleoperación tridimensional utilizando la cámara de profundidad Intel RealSense y MediaPipe Hands.

**Nota**
- Las dependencias utilizadas se encuentran en el archivo `requirements_telecontrol.txt`del Github.

## Bloque 1: Importación de Librerías y Modo de Respaldo (Simulación)
1. **Ecosistema de Visión y Cálculo**:
- `pyrealsense2 (rs)`: SDK oficial de Intel RealSense encargado de capturar los flujos de color (RGB) y mapa de profundidad (Z16), alinearlos espacialmente y proyectar píxeles 2D a coordenadas métricas 3D $[X, Y, Z]$.
- `numpy (np)`: Motor de álgebra lineal para el cálculo vectorizado de distancias euclidianas, diferencias de profundidad, promedios estadísticos en calibración y normalización de consignas.
- `cv2 (OpenCV)`: Renderizado de la interfaz gráfica (HUD), dibujo del esqueleto (wireframe) y captura de eventos de teclado en tiempo real.
- `mediapipe (mp)`: Red neuronal de inferencia ligera que detecta 21 nodos articulares (landmarks) sobre la imagen RGB de la mano.
2. **Tolerancia a Fallos** (`MANO_CONECTADA`): El bloque `try-except` intenta cargar el controlador InspireHand desde el módulo local `inspire_rh56dftp.py`. Si el archivo no está presente en el entorno, la bandera `MANO_CONECTADA` pasa a `False`, permitiendo depurar toda la etapa de visión artificial y matemáticas en Modo Simulación sin interrumpir la ejecución.
3. **Filtro de Consola**: Se suprimen las advertencias de depreciación internas de `google.protobuf` generadas por MediaPipe para mantener limpia la salida estándar de Jupyter Lab. 

In [1]:
import pyrealsense2 as rs
import numpy as np
import cv2
import mediapipe as mp
import warnings
import csv
import os
import time

try:
    from inspire_rh56dftp import InspireHand
    MANO_CONECTADA = True
except ImportError:
    MANO_CONECTADA = False
    print("⚠ No se encontró 'inspire_rh56dftp.py'. El script correrá en modo Simulación.")

# Silenciar advertencia de Protobuf
warnings.filterwarnings("ignore", category=UserWarning, module="google.protobuf.symbol_database")

## Bloque 2: Inicialización de Seguridad y Limpieza de Caché del Hardware
Cuando se interrumpe un script previo o se ejecutan rutinas con velocidades/fuerzas personalizadas, la memoria RAM del controlador interno de la mano puede retener restricciones antiguas. Este bloque se ejecuta al arranque mediante una instancia temporal (`_mano_init`) para purgar el estado del hardware:   
| Registro Modbus | Parámetro | Valor Inyectado | Propósito Técnico | 
|---|---|---|---|
| `1006` | Reset de Parámetros | `[1]` | Ejecuta un restablecimiento de fábrica en el controlador y limpia estados residuales (requiere 1.0 s de estabilización). |   
| `1498` | Límite de Fuerza (`REG_FORCE_SET`) | `[300, 300, 300, 300, 500, 300]` | Limita el torque operativo a $300\,\text{g}$ en los dedos principales/rotación y $500\,\text{g}$ en la flexión del pulgar para evitar daños por colisión y lograr un movimiento fluido en la teleoperación. | 
| `1044` | Fuerza por Defecto (`REG_DEFAULT_FORCE_SET`) | `[300, 300, 300, 300, 400, 300]`| Fija el umbral de protección predeterminado a nivel de firmware (encapsulado en `try-except` por compatibilidad de versiones). |   
| `1522` | Velocidad (`REG_SPEED_SET`) | `[1000, 1000, 1000, 1000, 1000, 1000]`| Restaura los 6 grados de libertad (`DOF`) a su velocidad máxima (`1000`) para garantizar respuesta inmediata durante el telecontrol. |

In [2]:
try:
    print("\nInyectando perfil de Fuerza Segura a la memoria de la mano...")
    _mano_init = InspireHand(ip="192.168.123.211")
    print("Ejecutando restablecimiento de fábrica...")
    _mano_init._write_and_ack(1006, [1]) 
    time.sleep(1.0)
    
    # Vector de fuerza segura para todos los dedos (Rango 0-1000/1500)
    fuerza_segura = [100, 100, 100, 100, 200, 200]
    
    # 1. Asignar Fuerza Segura Límite (Registro 1498)
    _mano_init._write_and_ack(1498, fuerza_segura)
    time.sleep(0.1)
    
    # 2. Asignar Fuerza Segura por Defecto (Registro 1044)
    try:
        _mano_init._write_and_ack(1044, fuerza_segura)
        time.sleep(0.1)
    except Exception:
        pass # Se ignora si tu versión de firmware omite este registro
        
    velocidades_maximas = [1000, 1000, 1000, 1000, 1000, 1000]
    print("Restaurando velocidad a máxima...")
    _mano_init._write_and_ack(1522, velocidades_maximas)
    time.sleep(0.1)
        
    print("✓ Fuerza segura configurada exitosamente. Lista para operar.\n")
except Exception as e:
    print(f"⚠ Aviso durante la inicialización de seguridad: {e}\n")


Inyectando perfil de Fuerza Segura a la memoria de la mano...
Ejecutando restablecimiento de fábrica...
Restaurando velocidad a máxima...
✓ Fuerza segura configurada exitosamente. Lista para operar.



## Bloque 3: Configuración de Red, Archivos CSV y Funciones de Control Modbus
1. **Persistencia y Estructura de Datos**:
- Se define la dirección IP estática de la mano (`192.168.123.211`) y el vector de reposo `ABIERTO` con los 6 actuadores completamente extendidos (`1000`).
- Mediante `os.path.expanduser` y `os.makedirs(..., exist_ok=True)`, el script garantiza que existan las carpetas locales para almacenar los archivos `gestos_guardados_3d.csv` y `rutinas_guardadas_3d.csv` sin arrojar errores si ya fueron creadas.
  
2. **Parámetros Operativos de Fuerza** (`FUERZA_SEGURA`):
- Define el vector `[100, 100, 100, 100, 250, 250]` que se inyectará al iniciar la función `main()`, otorgando $500\,\text{g}$ de límite a la flexión del pulgar (`pos(4)`) y $300\,\text{g}$ al resto de los actuadores.
  
3. **Función** `enviar_angulos(mano, angulos, timeout=5.0, tolerancia=40)`:
- Se utiliza para movimientos puntuales controlados (apertura inicial, cierre seguro al apagar el programa y reproducción de gestos/rutinas desde el menú).
- Envía el comando mediante `mano.send_raw(angulos)` y entra en un bucle de verificación leyendo los 6 registros de posición (`read_holding(REG_ANGLE_ACT, 6)` en la dirección `1486`) cada $100\,\text{ms}$ hasta confirmar que el error absoluto en cada motor es menor o igual a `40` unidades Modbus, o hasta que expire el `timeout` de $5.0\,\text{s}$.   

In [3]:
IP_MANO = "192.168.123.211"
ABIERTO = [1000, 1000, 1000, 1000, 1000, 1000]

DIRECTORIO_GUARDADO = os.path.expanduser("~/Documents/Jupyter/ManoRH56DFTP-2R/Gestos")
DIRECTORIO_RUTINAS = os.path.expanduser("~/Documents/Jupyter/ManoRH56DFTP-2R/Rutinas")
os.makedirs(DIRECTORIO_GUARDADO, exist_ok=True)
os.makedirs(DIRECTORIO_RUTINAS, exist_ok=True)

ARCHIVO_GESTOS_3D = os.path.join(DIRECTORIO_GUARDADO, "gestos_guardados_3d.csv")
ARCHIVO_RUTINAS_3D = os.path.join(DIRECTORIO_RUTINAS, "rutinas_guardadas_3d.csv")
CABECERAS_GESTOS = ["Nombre_Gesto", "Ang_Meñique", "Ang_Anular", "Ang_Corazón", "Ang_Índice", "Ang_Flexión_pulgar", "Ang_Rotación_pulgar"]
CABECERAS_RUTINAS = ["Nombre_Rutina", "Secuencia_Indices", "Secuencia_Nombres"]

REG_ANGLE_ACT = 1486
REG_FORCE_SET = 1498
FUERZA_SEGURA = [100, 100, 100, 100, 250, 250]

def enviar_angulos(mano, angulos, timeout=5.0, tolerancia=40):
    if not MANO_CONECTADA or mano is None:
        print(f" (Simulación) Enviando ángulos a la mano: {angulos}")
        time.sleep(1)
        return True
        
    mano.send_raw(angulos)
    inicio = time.time()
    while time.time() - inicio < timeout:
        try:
            angulos_act = list(mano.read_holding(REG_ANGLE_ACT, 6))
            if all(abs(act - obj) <= tolerancia for act, obj in zip(angulos_act, angulos)):
                return True
        except Exception:
            pass 
        time.sleep(0.1)
    print(f" ⚠ Tiempo agotado ({timeout}s) para alcanzar la pose exacta.")
    return False

## Bloque 4: Gestión de base de datos (.csv)
- `cargar_gestos()` y `cargar_rutinas()`: Verifican la existencia de los archivos `.csv`, abren el flujo en modo lectura (`'r'`), omiten la primera fila correspondiente a los encabezados usando `next(reader)` (protegido con `StopIteration si el archivo está vacío) y retornan una lista limpia con las filas válidas.
- `exportar_rutina(nombre, indices, gestos_guardados)`: Recibe una lista de índices numéricos que apuntan a gestos previamente almacenados, construye tanto la cadena de índices (`"0,2,1"`) como la cadena legible de transición (`"GestoA -> GestoC -> GestoB"`), y las añade al final (`mode='a'`) del archivo `rutinas_guardadas_3d.csv`.   

In [4]:
def cargar_gestos():
    if not os.path.exists(ARCHIVO_GESTOS_3D):
        return []
    with open(ARCHIVO_GESTOS_3D, mode='r') as file:
        reader = csv.reader(file)
        try:
            next(reader) 
        except StopIteration:
            pass
        return [row for row in reader if row]

def cargar_rutinas():
    if not os.path.exists(ARCHIVO_RUTINAS_3D):
        return []
    with open(ARCHIVO_RUTINAS_3D, mode='r') as file:
        reader = csv.reader(file)
        try:
            next(reader) 
        except StopIteration:
            pass
        return [row for row in reader if row and len(row) >= 2]

def exportar_rutina(nombre, indices, gestos_guardados):
    existe = os.path.exists(ARCHIVO_RUTINAS_3D)
    sec_idx = ",".join(str(i) for i in indices)
    sec_nom = " -> ".join([gestos_guardados[i][0] for i in indices])
    with open(ARCHIVO_RUTINAS_3D, mode='a', newline='') as file:
        writer = csv.writer(file)
        if not existe:
            writer.writerow(CABECERAS_RUTINAS)
        writer.writerow([nombre, sec_idx, sec_nom])
    print(f" ✓ Rutina '{nombre}' guardada.")

## Bloque 5: Calibración Biométrica 3D en 4 Fases (`ejecutar_calibracion`)

### 1. Extracción de Nodos, Filtro Espacial de Vecindad y Deproyección 3D
El sistema selecciona 11 nodos estratégicos del modelo de MediaPipe: `nodos = [0, 2, 4, 5, 8, 9, 12, 13, 16, 17, 20]` únicamente cuando detecta la etiqueta `"Left"` (mano derecha física frente a cámara sin espejo):   
- **Nodos de falanges principales**: Base y punta del Índice (`5` y `8`), Corazón (`9` y `12`), Anular (`13` y `16`) y Meñique (`17` y `20`).
- Nodos del pulgar y palma: Muñeca (`0`), base del pulgar (`2`) y yema del pulgar (`4`).
- **Filtro de Vecindad** $5 \times 5$: Si el sensor infrarrojo devuelve `z == 0` en el borde de un dedo por sombra óptica, el algoritmo barre una ventana de `[-2, -1, 0, 1, 2]` píxeles en $X$ e $Y$ alrededor del nodo y rescata la primera lectura de profundidad válida `(z_vecino > 0)`.
- **Deproyección Métrica**: Con la distancia $z$ en metros y los parámetros intrínsecos de la lente (`intrinsics`), la instrucción `rs.rs2_deproject_pixel_to_point(intrinsics, [cx, cy], z)` transforma el píxel `[cx, cy]` en una coordenada física tridimensional $[X, Y, Z]$ en metros.

### 2. Vector de Magnitudes Físicas (`D_actual`)
Con los 11 puntos 3D validados, se construye el vector de 6 elementos combinando distancias euclidianas 3D y diferencial de profundidad:
```bash
D_actual = np.array([
    np.linalg.norm(puntos_3d[20] - puntos_3d[17]), # pos(0): Meñique
    np.linalg.norm(puntos_3d[16] - puntos_3d[13]), # pos(1): Anular
    np.linalg.norm(puntos_3d[12] - puntos_3d[9]),  # pos(2): Corazón
    np.linalg.norm(puntos_3d[8] - puntos_3d[5]),   # pos(3): Índice
    np.linalg.norm(puntos_3d[4] - puntos_3d[5]),   # pos(4): Flexión pulgar (Euclidiana nodos 4 y 5)
    abs(puntos_3d[4][2] - puntos_3d[5][2])         # pos(5): Rotación pulgar (|Z_4 - Z_5|)
])
```
- Motores `0` a `4`: Se calculan mediante la norma euclidiana tridimensional $\Vert{}\mathbf{P}_a - \mathbf{P}_b\Vert{}_2 = \sqrt{\Delta X^2 + \Delta Y^2 + \Delta Z^2}$.
- Motor `5` (Rotación del Pulgar): Se calcula exclusivamente como el valor absoluto de la diferencia en el eje de profundidad $Z$ entre la yema del pulgar (nodo `4`) y la base del dedo índice (nodo `5`): $\Delta Z = \vert{}Z_4 - Z_5\vert{}$.

### 3, Máquina de Estados de las 4 Fases de Calibración
Cada fase requiere presionar `c` 3 veces para promediar las muestras `(np.mean(..., axis=0))` y eliminar el ruido de captura:   
1. Fase 1 (`MÁXIMA` - Mano Abierta): Captura la extensión máxima de los dedos `0` al `4` y la guarda en `D_max[0:5]`. Como con la mano abierta paralela a la cámara el pulgar y la base del índice están en el mismo plano de profundidad, esta postura registra la menor diferencia en $Z$, por lo que se almacena en `D_min[5]` (que representará el valor `1000` de rotación abierta).
2. Fase 2 (`MÍNIMA_DEDOS` - Puño 4 Dedos): Con los cuatro dedos largos cerrados, almacena sus distancias mínimas en `D_min[0:4]` sin alterar los valores del pulgar.
3. Fase 3 (`MÍNIMA_FLEX_PULGAR` - Flexión Pulgar): Con la mano abierta y acercando el pulgar hacia la base del índice, registra la distancia euclidiana mínima entre los nodos `4` y `5` en `D_min[4]`.
4. Fase 4 (`MÍNIMA_ROT_PULGAR` - Rotación Pulgar Z): Al rotar el pulgar hacia adelante/adentro en oposición a la palma, se genera la mayor separación de profundidad $\Delta Z$ entre la yema del pulgar (`4`) y la base del índice (`5`). Este valor máximo se guarda en `D_max[5]` (que tras la inversión equivaldrá a `0` en el actuador de rotación). 

In [5]:
def ejecutar_calibracion(pipeline, align, hands, intrinsics):
    modo = "MÁXIMA"
    muestras_max = []
    muestras_min_dedos = []
    muestras_min_flex_p = []
    muestras_min_rot_p = []
    D_max, D_min = np.zeros(6), np.zeros(6)

    print("\n--- INICIANDO CALIBRACIÓN 3D (4 FASES) ---")
    print("Mira la ventana de video y sigue las instrucciones:")
    print("1. Mano Abierta (D_max[0:5] y menor diferencia Z en D_min[5] -> 1000).")
    print("2. Puño Cerrado (D_min[0:4] para los 4 dedos principales).")
    print("3. Flexión Mínima Pulgar: acerca la yema del pulgar a la yema del índice (D_min[4]).")
    print("4. Rotación Mínima Pulgar: rota el pulgar generando la mayor diferencia Z con base del índice (D_max[5] -> 0).")
    
    while True:
        frames = pipeline.wait_for_frames()
        aligned_frames = align.process(frames)
        color_frame = aligned_frames.get_color_frame()
        depth_frame = aligned_frames.get_depth_frame()
        if not color_frame or not depth_frame:
            continue
        
        color_image = np.asanyarray(color_frame.get_data())
        image_rgb = cv2.cvtColor(color_image, cv2.COLOR_BGR2RGB)
        results = hands.process(image_rgb)
        
        D_actual = np.zeros(6)
        mano_detectada = False
        prof_valida = True

        if results.multi_hand_landmarks and results.multi_handedness:
            for hand_landmarks, handedness in zip(results.multi_hand_landmarks, results.multi_handedness):
                if handedness.classification[0].label == "Left":
                    mano_detectada = True
                    h, w, _ = color_image.shape
                    nodos = [0, 2, 4, 5, 8, 9, 12, 13, 16, 17, 20]
                    puntos_3d = {}
                    pixeles = {}
                    
                    for idx in nodos:
                        lm = hand_landmarks.landmark[idx]
                        cx, cy = max(0, min(int(lm.x * w), w - 1)), max(0, min(int(lm.y * h), h - 1))
                        pixeles[idx] = (cx, cy)
                        z = depth_frame.get_distance(cx, cy)
                        
                        # --- FILTRO DE VECINDAD PARA BORDES ---
                        if z == 0:
                            for dx in [-2, -1, 0, 1, 2]:
                                for dy in [-2, -1, 0, 1, 2]:
                                    nx, ny = max(0, min(cx + dx, w - 1)), max(0, min(cy + dy, h - 1))
                                    z_vecino = depth_frame.get_distance(nx, ny)
                                    if z_vecino > 0:
                                        z = z_vecino
                                if z > 0:
                                    break
                        
                        if z > 0:
                            puntos_3d[idx] = np.array(rs.rs2_deproject_pixel_to_point(intrinsics, [cx, cy], z))
                        else:
                            prof_valida = False

                    # --- DIBUJO DEL WIREFRAME EN CALIBRACIÓN ---
                    for idx, (cx, cy) in pixeles.items():
                        cv2.circle(color_image, (cx, cy), 6, (255, 0, 0), cv2.FILLED)
                    
                    if len(pixeles) == 11:
                        cv2.line(color_image, pixeles[0], pixeles[5], (200, 200, 200), 2)
                        cv2.line(color_image, pixeles[0], pixeles[9], (200, 200, 200), 2)
                        cv2.line(color_image, pixeles[0], pixeles[13], (200, 200, 200), 2)
                        cv2.line(color_image, pixeles[0], pixeles[17], (200, 200, 200), 2)
                        cv2.line(color_image, pixeles[0], pixeles[2], (200, 200, 200), 2)
                        cv2.line(color_image, pixeles[2], pixeles[4], (200, 200, 200), 2)
                        cv2.line(color_image, pixeles[5], pixeles[8], (0, 255, 0), 3)    # Índice
                        cv2.line(color_image, pixeles[9], pixeles[12], (0, 255, 0), 3)   # Corazón
                        cv2.line(color_image, pixeles[13], pixeles[16], (0, 255, 0), 3)  # Anular
                        cv2.line(color_image, pixeles[17], pixeles[20], (0, 255, 0), 3)  # Meñique
                        cv2.line(color_image, pixeles[5], pixeles[4], (0, 255, 255), 3)  # Pulgar (Nodos 4 y 5)

                    if prof_valida:
                        D_actual = np.array([
                            np.linalg.norm(puntos_3d[20] - puntos_3d[17]), # pos(0): Meñique
                            np.linalg.norm(puntos_3d[16] - puntos_3d[13]), # pos(1): Anular
                            np.linalg.norm(puntos_3d[12] - puntos_3d[9]),  # pos(2): Corazón
                            np.linalg.norm(puntos_3d[8] - puntos_3d[5]),   # pos(3): Índice
                            np.linalg.norm(puntos_3d[4] - puntos_3d[5]),   # pos(4): Flexión pulgar (Euclidiana nodos 4 y 5)
                            abs(puntos_3d[4][2] - puntos_3d[5][2])         # pos(5): Rotación pulgar (|Z_4 - Z_5|)
                        ])
                    else:
                        cv2.putText(color_image, "Mueve la mano, borde perdiendo profundidad", (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)

                    if modo == "MÁXIMA":
                        cv2.putText(color_image, f"1. Abierta ({len(muestras_max)}/3) - Presiona 'c'", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
                    elif modo == "MÍNIMA_DEDOS":
                        cv2.putText(color_image, f"2. Puno 4 Dedos ({len(muestras_min_dedos)}/3) - Presiona 'c'", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
                    elif modo == "MÍNIMA_FLEX_PULGAR":
                        cv2.putText(color_image, f"3. Flexion Pulgar ({len(muestras_min_flex_p)}/3) - Presiona 'c'", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)
                    elif modo == "MÍNIMA_ROT_PULGAR":
                        cv2.putText(color_image, f"4. Rotacion Pulgar Z ({len(muestras_min_rot_p)}/3) - Presiona 'c'", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 255), 2)

        cv2.imshow('Calibracion 3D', color_image)

        key = cv2.waitKey(1) & 0xFF
        if key == ord('c') and mano_detectada and prof_valida:
            if modo == "MÁXIMA":
                muestras_max.append(D_actual.copy())
                print(f"✓ Medida Máxima {len(muestras_max)}/3 capturada.")
                if len(muestras_max) == 3:
                    promedio_abierta = np.mean(np.array(muestras_max), axis=0)
                    D_max[0:5] = promedio_abierta[0:5]
                    D_min[5] = promedio_abierta[5]  # Menor diferencia en Z corresponde a 1000
                    print("✓ FASE 1 completada. -> FASE 2: Cierra el puño (4 dedos principales) y presiona 'c'.")
                    modo = "MÍNIMA_DEDOS"
                    
            elif modo == "MÍNIMA_DEDOS":
                muestras_min_dedos.append(D_actual.copy())
                print(f"✓ Medida Mínima Dedos {len(muestras_min_dedos)}/3 capturada.")
                if len(muestras_min_dedos) == 3:
                    D_min[0:4] = np.mean(np.array(muestras_min_dedos), axis=0)[0:4]
                    print("✓ FASE 2 completada. -> FASE 3: Acerca el pulgar al índice (mínima flexión) y presiona 'c'.")
                    modo = "MÍNIMA_FLEX_PULGAR"
                    
            elif modo == "MÍNIMA_FLEX_PULGAR":
                muestras_min_flex_p.append(D_actual.copy())
                print(f"✓ Medida Mínima Flexión Pulgar {len(muestras_min_flex_p)}/3 capturada.")
                if len(muestras_min_flex_p) == 3:
                    D_min[4] = np.mean(np.array(muestras_min_flex_p), axis=0)[4]
                    print("✓ FASE 3 completada. -> FASE 4: Rota el pulgar (mayor diferencia Z con base del índice) y presiona 'c'.")
                    modo = "MÍNIMA_ROT_PULGAR"
                    
            elif modo == "MÍNIMA_ROT_PULGAR":
                muestras_min_rot_p.append(D_actual.copy())
                print(f"✓ Medida Mínima Rotación Pulgar {len(muestras_min_rot_p)}/3 capturada.")
                if len(muestras_min_rot_p) == 3:
                    D_max[5] = np.mean(np.array(muestras_min_rot_p), axis=0)[5]  # Mayor diferencia en Z corresponde a 0
                    print("\n✓ CALIBRACIÓN EN 4 FASES COMPLETADA CON ÉXITO.")
                    print(f"D_max = np.array([{', '.join(f'{v:.4f}' for v in D_max)}])")
                    print(f"D_min = np.array([{', '.join(f'{v:.4f}' for v in D_min)}])")
                    cv2.destroyWindow('Calibracion 3D')
                    return D_max, D_min
                    
        elif key == ord('q'):
            print("⚠ Calibración cancelada. Se usarán valores por defecto.")
            cv2.destroyWindow('Calibracion 3D')
            return None, None

## Bloque 6: Programa Principal (`main`) — Configuración Inicial y Parámetros de Control
- **Inicialización de la Mano y Cámara**: Establece el enlace Modbus TCP definitivo, aplica el vector `FUERZA_SEGURA` (`[300, 300, 300, 300, 500, 300]`) sobre el registro `1498` y lleva la mano a la postura `ABIERTO`. Posteriormente inicializa MediaPipe (`max_num_hands=1`, `min_detection_confidence=0.7`) y levanta el flujo de la cámara RealSense a $640 \times 480$ píxeles y $30\,\text{FPS}$, alineando el mapa de profundidad al cuadro de color (`rs.align(rs.stream.color)`).
- **Matrices Predeterminadas** (`D_max` y `D_min`): Permiten operar de inmediato con la Opción 1 del menú sin necesidad de calibrar en cada sesión. En `pos(5)`, `D_max[5] = 0.0650` ($6.5\,\text{cm}$) representa la diferencia máxima de profundidad para rotación `0`, y `D_min[5] = 0.0150` ($1.5\,\text{cm}$) la diferencia mínima para rotación `1000`.

### Variables de Filtrado para el Pulgar y Zonas Muertas de Movimiento
```bash 
#Sensibilidades independientes para cada grupo de dedos:
UMBRAL_MOVIMIENTO  = 100    # Para dedos 0, 1, 2 y 3 (Meñique a Índice)
UMBRAL_FLEX_PULGAR = 15     # Más sensible para detectar la flexión pos(4)
UMBRAL_ROTACION    = 50     # Exclusivo de la rotación pos(5)

UMBRAL_ROT_CERCANO_CERO = 150  
UMBRAL_OCLUSION_INDICE  = 10  
HIST_FLEXION = 15              

rotacion_retenida = 1000       
flexion_anterior = 1000        
rotacion_bloqueada = False
```
    
- `UMBRAL_MOVIMIENTO = 100`: Banda muerta amplia para los 4 dedos largos (`0:4`), filtrando temblores del esqueleto sin perder cierres francos.
- `UMBRAL_FLEX_PULGAR = 15`: Banda muerta de alta sensibilidad exclusiva para la flexión del pulgar (`pos(4)`), garantizando que recorridos cortos entre los nodos `4` y `5` se reflejen fielmente en la mano robótica.
- `UMBRAL_ROTACION = 50`: Escalón mínimo exigido para actualizar la consigna de rotación del pulgar (`pos(5)`).
- `UMBRAL_OCLUSION_INDICE = 10`: Umbral crítico de cierre del dedo índice (`pos(3)`). Cuando el índice desciende a `<= 10`, oculta físicamente el nodo `5` frente a la cámara.
- `UMBRAL_ROT_CERCANO_CERO = 150` y `HIST_FLEXION = 15`: Parámetros de estados que gobiernan el bloqueo de rotación cuando el pulgar ya ha rotado cerca de `0` y comienza a flexionarse.   

## Bloque 7: Bucle de Seguimiento Continuo, Normalización e Inversión Cinemática
Dentro del bucle `while True`:, una vez extraídos los nodos 3D y dibujado el wireframe sobre la imagen, se ejecuta el bloque matemático central:

```bash
# Normalización estándar [0.0 a 1.0]
denominador = np.where(np.abs(D_max - D_min) < 1e-6, 1e-6, D_max - D_min)
Apertura_decimal = np.clip((D_actual - D_min) / denominador, 0.0, 1.0)

# Inversión para pos(5): D_max -> 0 y D_min -> 1000
Apertura_decimal[5] = 1.0 - Apertura_decimal[5]

vector_crudo = (Apertura_decimal * 1000).astype(int)
indice_actual = vector_crudo[3] # Posición del índice
flex_actual = vector_crudo[4]   # Posición de flexión del pulgar
rot_calculada = vector_crudo[5] # Posición de rotación calculada
```

#### **Explicación matemática**
1. **Protección contra División por Cero**: `np.where(np.abs(D_max - D_min) < 1e-6, 1e-6, D_max - D_min)` evita indeterminaciones si algún eje tuvo valores idénticos durante la calibración.
2. **Normalización Lineal Acotada**: Se proyecta `D_actual` al intervalo $[0.0, 1.0]$ mediante:$$\text{Apertura}_i = \text{clip}\left(\frac{D_{\text{actual}, i} - D_{\min, i}}{D_{\max, i} - D_{\min, i}},\, 0.0,\, 1.0\right)$$
3. **Inversión del Motor de Rotación** (`pos(5)`): Dado que en la rotación del pulgar una mayor diferencia de profundidad ($\Delta Z \to D_{\max}[5]$) corresponde físicamente al cierre del giro (valor Modbus `0`) y una menor diferencia ($\Delta Z \to D_{\min}[5]$) corresponde a la mano abierta (valor Modbus `1000`), se aplica el complemento:$$\text{Apertura}_5 \leftarrow 1.0 - \text{Apertura}_5$$
4. **Escalado Entero**: Se multiplica por `1000` y se convierte a entero de 32 bits (`vector_crudo`) para obtener el rango nativo `[0, 1000]` del controlador Inspire.   

In [6]:
def main():
    mano = None
    if MANO_CONECTADA:
        print(f"Conectando a la mano en {IP_MANO} ...")
        try:
            mano = InspireHand(ip=IP_MANO)
            
            # --- INYECCIÓN DE FUERZA SEGURA ---
            print("Configurando fuerza segura (300|500(4)) en todos los motores...")
            mano._write_and_ack(REG_FORCE_SET, FUERZA_SEGURA)
            time.sleep(0.1)
            # ----------------------------------
            
            enviar_angulos(mano, ABIERTO)
        except Exception as e:
            print(f"Error conectando a la mano: {e}. Pasando a simulación.")

    # Matriz por defecto (pos(4) entre nodos 4 y 5 | pos(5): D_max[5] -> 0 y D_min[5] -> 1000)
    D_max = np.array([0.1045, 0.1245, 0.1312, 0.1197, 0.1239, 0.0650])
    D_min = np.array([0.0261, 0.0280, 0.0250, 0.0218, 0.0474, 0.0150]) 
            
    mp_hands = mp.solutions.hands
    hands = mp_hands.Hands(max_num_hands=1, min_detection_confidence=0.7)

    pipeline = rs.pipeline()
    config = rs.config()
    config.enable_stream(rs.stream.color, 640, 480, rs.format.bgr8, 30)
    config.enable_stream(rs.stream.depth, 640, 480, rs.format.z16, 30)
    align = rs.align(rs.stream.color)
    pipeline.start(config)

    perfil_camara = pipeline.get_active_profile()
    intrinsics = perfil_camara.get_stream(rs.stream.depth).as_video_stream_profile().get_intrinsics()

    # --- MENÚ DE INICIO ---
    print("\n" + "="*40)
    print(" BIENVENIDO AL SISTEMA DE VISIÓN 3D")
    print("="*40)
    print("1. Iniciar cámara de ejecución (Usar calibración predeterminada)")
    print("2. Calibrar mi mano en 4 fases (Recomendado para mayor precisión)")
    opcion = input("Seleccione una opción (1 o 2): ").strip()

    if opcion == '2':
        D_max_nuevo, D_min_nuevo = ejecutar_calibracion(pipeline, align, hands, intrinsics)
        if D_max_nuevo is not None:
            D_max = D_max_nuevo
            D_min = D_min_nuevo
            print("Nuevas matrices en RAM. Transicionando al modo de Ejecución...")

    print("\n=== CÁMARA DE EJECUCIÓN CONTINUA ===")
    print(" 'e' : Iniciar / Pausar Seguimiento en Tiempo Real (Toggle)")
    print(" 's' : Guardar gesto actual en CSV")
    print(" 'r' : Abrir Menú de Rutinas (Consola)")
    print(" 'q' : Salir")

    # Variables para el tracking continuo
    seguimiento_continuo = False
    ultimo_envio = 0
    FRECUENCIA_ENVIO = 0.05  # Tasa de envío (20Hz)
    
    # --- AJUSTES DE UMBRALES SEGMENTADOS ---
    ultimo_vector_enviado = np.array(ABIERTO)
    
    # Sensibilidades independientes para cada grupo de dedos:
    UMBRAL_MOVIMIENTO  = 100    # Para dedos 0, 1, 2 y 3 (Meñique a Índice)
    UMBRAL_FLEX_PULGAR = 15    # Más sensible para detectar la flexión pos(4)
    UMBRAL_ROTACION    = 50    # Exclusivo de la rotación pos(5)
    
    UMBRAL_ROT_CERCANO_CERO = 150  
    UMBRAL_OCLUSION_INDICE  = 10  
    HIST_FLEXION = 15              

    rotacion_retenida = 1000       
    flexion_anterior = 1000        
    rotacion_bloqueada = False     

    try:
        while True:
            frames = pipeline.wait_for_frames()
            aligned_frames = align.process(frames)
            
            color_frame = aligned_frames.get_color_frame()
            depth_frame = aligned_frames.get_depth_frame()
            if not color_frame or not depth_frame:
                continue
            
            color_image = np.asanyarray(color_frame.get_data())
            image_rgb = cv2.cvtColor(color_image, cv2.COLOR_BGR2RGB)
            results = hands.process(image_rgb)
            
            vector_modbus_actual = None
            mano_en_pantalla = False
            profundidad_valida = True

            if results.multi_hand_landmarks and results.multi_handedness:
                for hand_landmarks, handedness in zip(results.multi_hand_landmarks, results.multi_handedness):
                    if handedness.classification[0].label == "Left":
                        mano_en_pantalla = True
                        h, w, _ = color_image.shape
                        nodos = [0, 2, 4, 5, 8, 9, 12, 13, 16, 17, 20]
                        puntos_3d = {}
                        pixeles_dibujo = {}
                        
                        for idx in nodos:
                            lm = hand_landmarks.landmark[idx]
                            cx, cy = max(0, min(int(lm.x * w), w - 1)), max(0, min(int(lm.y * h), h - 1))
                            pixeles_dibujo[idx] = (cx, cy)
                            
                            z_metros = depth_frame.get_distance(cx, cy)
                            
                            # --- FILTRO DE VECINDAD ---
                            if z_metros == 0:
                                for dx in [-2, -1, 0, 1, 2]:
                                    for dy in [-2, -1, 0, 1, 2]:
                                        nx, ny = max(0, min(cx + dx, w - 1)), max(0, min(cy + dy, h - 1))
                                        z_vecino = depth_frame.get_distance(nx, ny)
                                        if z_vecino > 0:
                                            z_metros = z_vecino
                                    if z_metros > 0:
                                        break
                            # --------------------------
                            
                            if z_metros > 0:
                                puntos_3d[idx] = np.array(rs.rs2_deproject_pixel_to_point(intrinsics, [cx, cy], z_metros))
                            else:
                                profundidad_valida = False

                        # Dibujo del Wireframe
                        for idx, (cx, cy) in pixeles_dibujo.items():
                            cv2.circle(color_image, (cx, cy), 6, (255, 0, 0), cv2.FILLED)
                        
                        if len(pixeles_dibujo) == 11:
                            cv2.line(color_image, pixeles_dibujo[0], pixeles_dibujo[5], (200, 200, 200), 2)
                            cv2.line(color_image, pixeles_dibujo[0], pixeles_dibujo[9], (200, 200, 200), 2)
                            cv2.line(color_image, pixeles_dibujo[0], pixeles_dibujo[13], (200, 200, 200), 2)
                            cv2.line(color_image, pixeles_dibujo[0], pixeles_dibujo[17], (200, 200, 200), 2)
                            cv2.line(color_image, pixeles_dibujo[0], pixeles_dibujo[2], (200, 200, 200), 2)
                            cv2.line(color_image, pixeles_dibujo[2], pixeles_dibujo[4], (200, 200, 200), 2)
                            cv2.line(color_image, pixeles_dibujo[5], pixeles_dibujo[8], (0, 255, 0), 3)    # Índice
                            cv2.line(color_image, pixeles_dibujo[9], pixeles_dibujo[12], (0, 255, 0), 3)   # Corazón
                            cv2.line(color_image, pixeles_dibujo[13], pixeles_dibujo[16], (0, 255, 0), 3)  # Anular
                            cv2.line(color_image, pixeles_dibujo[17], pixeles_dibujo[20], (0, 255, 0), 3)  # Meñique
                            cv2.line(color_image, pixeles_dibujo[5], pixeles_dibujo[4], (0, 255, 255), 3)  # Pulgar (Nodos 4 y 5)

                        if profundidad_valida:
                            D_actual = np.array([
                                np.linalg.norm(puntos_3d[20] - puntos_3d[17]), # pos(0): Meñique
                                np.linalg.norm(puntos_3d[16] - puntos_3d[13]), # pos(1): Anular
                                np.linalg.norm(puntos_3d[12] - puntos_3d[9]),  # pos(2): Corazón
                                np.linalg.norm(puntos_3d[8] - puntos_3d[5]),   # pos(3): Índice
                                np.linalg.norm(puntos_3d[4] - puntos_3d[5]),   # pos(4): Flexión pulgar (Euclidiana nodos 4 y 5)
                                abs(puntos_3d[4][2] - puntos_3d[5][2])         # pos(5): Rotación pulgar (|Z_4 - Z_5|)
                            ])
                            
                            # Normalización estándar [0.0 a 1.0]
                            denominador = np.where(np.abs(D_max - D_min) < 1e-6, 1e-6, D_max - D_min)
                            Apertura_decimal = np.clip((D_actual - D_min) / denominador, 0.0, 1.0)
                            
                            # Inversión para pos(5): D_max -> 0 y D_min -> 1000
                            Apertura_decimal[5] = 1.0 - Apertura_decimal[5]
                            
                            vector_crudo = (Apertura_decimal * 1000).astype(int)
                            indice_actual = vector_crudo[3] # Posición del índice
                            flex_actual = vector_crudo[4]   # Posición de flexión del pulgar
                            rot_calculada = vector_crudo[5] # Posición de rotación calculada
                            
                            # Condiciones combinadas de bloqueo y forzado de rotación
                            delta_flex = flex_actual - flexion_anterior
                            bloqueo_por_oclusion = False
                        
                            if indice_actual <= UMBRAL_OCLUSION_INDICE:
                                rotacion_retenida = 1000
                                rotacion_bloqueada = True
                                bloqueo_por_oclusion = True
                            else:
                                if rotacion_retenida <= UMBRAL_ROT_CERCANO_CERO or rotacion_bloqueada:
                                    if delta_flex < -HIST_FLEXION:
                                        rotacion_bloqueada = True
                                    elif delta_flex > HIST_FLEXION:
                                        rotacion_bloqueada = False
                                else:
                                    rotacion_bloqueada = False

                            if abs(delta_flex) > HIST_FLEXION:
                                flexion_anterior = flex_actual

                            if not rotacion_bloqueada:
                                if abs(rot_calculada - rotacion_retenida) >= UMBRAL_ROTACION:
                                    rotacion_retenida = rot_calculada

                            vector_crudo[5] = rotacion_retenida
                            vector_modbus_actual = vector_crudo

            # --- LÓGICA DE SEGUIMIENTO EN TIEMPO REAL CON FILTROS INDEPENDIENTES ---
            if seguimiento_continuo and mano_en_pantalla and vector_modbus_actual is not None and profundidad_valida:
                if time.time() - ultimo_envio >= FRECUENCIA_ENVIO:
                    
                    # Desglosamos los deltas matemáticos para analizarlos independientemente
                    cambio_dedos_principales = np.max(np.abs(vector_modbus_actual[0:4] - ultimo_vector_enviado[0:4]))
                    cambio_flexion_pulgar    = np.abs(vector_modbus_actual[4] - ultimo_vector_enviado[4])
                    cambio_rotacion_pulgar   = np.abs(vector_modbus_actual[5] - ultimo_vector_enviado[5])
                    
                    # La condición es verdadera si ALGUNO de los actuadores vence a su propio umbral
                    if (cambio_dedos_principales > UMBRAL_MOVIMIENTO or 
                        cambio_flexion_pulgar > UMBRAL_FLEX_PULGAR or 
                        cambio_rotacion_pulgar > 0): # La rotación ya fue filtrada por su UMBRAL_ROTACION arriba
                        
                        if MANO_CONECTADA and mano is not None:
                            mano.send_raw(vector_modbus_actual.tolist())
                        
                        ultimo_vector_enviado = vector_modbus_actual.copy()
                        ultimo_envio = time.time()

            # --- RENDERIZADO VISUAL ---
            # Mostramos el vector_modbus_actual SIEMPRE para que los números en pantalla se actualicen 
            # de manera fluida, confirmando lo que lee la cámara, independientemente del envío Modbus.
            cv2.putText(color_image, "'e' Toggle Tracking | 's' Guardar | 'r' Menu Rutinas | 'q' Salir", (10, 30), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
            
            estado_txt = "EN VIVO" if seguimiento_continuo else "PAUSADO"
            color_estado = (0, 0, 255) if seguimiento_continuo else (0, 255, 0) 
            cv2.putText(color_image, f"Teleoperacion: {estado_txt}", (10, 60), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, color_estado, 2)

            if vector_modbus_actual is not None:
                if rotacion_bloqueada:
                    causa = "Oclusion Indice -> FORZADO 1000" if bloqueo_por_oclusion else "Flexion Pulgar -> BLOQUEADA"
                    estado_rot = causa
                else:
                    estado_rot = "ACTIVA"
                cv2.putText(color_image, f"Flex(4): {vector_modbus_actual[4]} | Rot(5): {vector_modbus_actual[5]} [{estado_rot}]", (10, 90),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 0), 2)
            
            cv2.imshow('Vision 3D - Control de Mano', color_image)

            key = cv2.waitKey(1) & 0xFF
            
            if key == ord('e'):
                seguimiento_continuo = not seguimiento_continuo
                if seguimiento_continuo:
                    ultimo_vector_enviado = np.array(ABIERTO) 
                print(f"\n>> MODO TELEOPERACIÓN: {'ACTIVADO' if seguimiento_continuo else 'PAUSADO'}")
            
            elif key == ord('s') and vector_modbus_actual is not None:
                seguimiento_continuo = False 
                print("\n--- GUARDAR NUEVO GESTO ---")
                nombre_gesto = input("Ingrese nombre para este gesto: ").strip()
                if nombre_gesto:
                    existe = os.path.exists(ARCHIVO_GESTOS_3D)
                    with open(ARCHIVO_GESTOS_3D, mode='a', newline='') as file:
                        writer = csv.writer(file)
                        if not existe:
                            writer.writerow(CABECERAS_GESTOS)
                        writer.writerow([nombre_gesto] + vector_modbus_actual.tolist())
                    print(f" ✓ Gesto '{nombre_gesto}' guardado.")
                    
            elif key == ord('r'):
                seguimiento_continuo = False 
                print("\n" + "="*40 + "\n         MENÚ DE RUTINAS 3D\n" + "="*40)
                print("1. Ejecutar un gesto guardado (.csv)")
                print("2. Crear y ejecutar NUEVA Rutina")
                print("3. Importar y ejecutar Rutina guardada")
                print("4. Volver a la cámara")
                opc = input("Seleccione una opción: ")
                gestos = cargar_gestos()
                
                if opc == '1' and gestos:
                    for i, g in enumerate(gestos):
                        print(f"{i}. {g[0]}")
                    try:
                        sel = int(input("Número de gesto (-1 cancelar): "))
                        if 0 <= sel < len(gestos):
                            angs = [int(v) for v in gestos[sel][1:7]]
                            enviar_angulos(mano, angs)
                    except ValueError:
                        pass
                        
                elif opc == '2' and gestos:
                    for i, g in enumerate(gestos):
                        print(f"{i}. {g[0]}")
                    try:
                        indices = [int(x) for x in input("\nSecuencia (ej. 0,2,1): ").split(',') if x.strip()]
                        validos = [i for i in indices if 0 <= i < len(gestos)]
                        if validos:
                            if input("¿Guardar rutina? (s/n): ").lower() == 's':
                                exportar_rutina(input("Nombre: "), validos, gestos)
                            if input("¿Ejecutar ahora? (s/n): ").lower() == 's':
                                for idx in validos:
                                    enviar_angulos(mano, [int(v) for v in gestos[idx][1:7]])
                                    time.sleep(1.0)
                    except ValueError:
                        pass

                elif opc == '3':
                    rutinas = cargar_rutinas()
                    if rutinas:
                        for i, r in enumerate(rutinas):
                            print(f"{i}. {r[0]} | Secuencia: {r[2]}")
                        try:
                            sel = int(input("Número de rutina (-1 cancelar): "))
                            if 0 <= sel < len(rutinas):
                                for idx in [int(x) for x in rutinas[sel][1].split(',') if x.strip()]:
                                    enviar_angulos(mano, [int(v) for v in gestos[idx][1:7]])
                                    time.sleep(1.0)
                        except ValueError:
                            pass
                print("\nRegresando a cámara...")
                
            elif key == ord('q'):
                break

    finally:
        pipeline.stop()
        cv2.destroyAllWindows()
        if mano is not None:
            enviar_angulos(mano, ABIERTO, timeout=2.0)
            mano.close()
        print("Cámara apagada y recursos liberados.")

## Bloque 8: Máquina de Estados de Protección para la Rotación del Pulgar

```bash
# Condiciones combinadas de bloqueo y forzado de rotación
delta_flex = flex_actual - flexion_anterior
bloqueo_por_oclusion = False

if indice_actual <= UMBRAL_OCLUSION_INDICE:
    rotacion_retenida = 1000
    rotacion_bloqueada = True
    bloqueo_por_oclusion = True
else:
    if rotacion_retenida <= UMBRAL_ROT_CERCANO_CERO or rotacion_bloqueada:
        if delta_flex < -HIST_FLEXION:
            rotacion_bloqueada = True
        elif delta_flex > HIST_FLEXION:
            rotacion_bloqueada = False
    else:
        rotacion_bloqueada = False

if abs(delta_flex) > HIST_FLEXION:
    flexion_anterior = flex_actual

if not rotacion_bloqueada:
    if abs(rot_calculada - rotacion_retenida) >= UMBRAL_ROTACION:
        rotacion_retenida = rot_calculada

vector_crudo[5] = rotacion_retenida
vector_modbus_actual = vector_crudo
```
Este bloque resuelve las dos fuentes principales de acoplamiento errático en visión monocular con profundidad:   
1. **Prioridad 1 — Oclusión por cierre del Dedo Índice** (`indice_actual <= 10`):
- Cuando el dedo índice se cierra casi por completo (`<= 10`), obstruye la línea de visión de la cámara hacia la base del índice (nodo `5`), provocando lecturas erráticas en $Z_5$.
- En este estado, el sistema fuerza `rotacion_retenida = 1000` (evitando colisiones mecánicas entre el pulgar y el índice cerrado) y `activa rotacion_bloqueada = True`.

2. **Prioridad 2 — Bloqueo por Flexión del Pulgar en Oposición** (`rotacion_retenida <= 150`):
- Cuando el usuario rota el pulgar hacia el centro de la palma (`rotacion_retenida <= 150`) y enseguida comienza a flexionarlo (`delta_flex < -15`, es decir, la flexión desciende de `1000` hacia `0`), la yema del pulgar cambia su distancia en $Z$ debido a la flexión y no a una rotación voluntaria.
- La máquina de estados activa `rotacion_bloqueada = True`, congelando la rotación en su valor cercano a `0` mientras el pulgar baja.
- Tan pronto el usuario extiende el pulgar de regreso (`delta_flex > 15`, subiendo de `0` hacia `1000`), la bandera se libera (`rotacion_bloqueada = False`) y se restablece el seguimiento normal de rotación.

3. **Filtro de Escalón** (`UMBRAL_ROTACION = 50`): Si la rotación está activa (`not rotacion_bloqueada`), `rotacion_retenida` solo se actualiza cuando el cambio respecto al valor calculado iguala o supera las `50` unidades Modbus.   

## Bloque 9: Envío Modbus en Tiempo Real con Filtros Independientes
```bash
# --- LÓGICA DE SEGUIMIENTO EN TIEMPO REAL CON FILTROS INDEPENDIENTES ---
if seguimiento_continuo and mano_en_pantalla and vector_modbus_actual is not None and profundidad_valida:
    if time.time() - ultimo_envio >= FRECUENCIA_ENVIO:
        
        # Desglosamos los deltas matemáticos para analizarlos independientemente
        cambio_dedos_principales = np.max(np.abs(vector_modbus_actual[0:4] - ultimo_vector_enviado[0:4]))
        cambio_flexion_pulgar    = np.abs(vector_modbus_actual[4] - ultimo_vector_enviado[4])
        cambio_rotacion_pulgar   = np.abs(vector_modbus_actual[5] - ultimo_vector_enviado[5])
        
        # La condición es verdadera si ALGUNO de los actuadores vence a su propio umbral
        if (cambio_dedos_principales > UMBRAL_MOVIMIENTO or 
            cambio_flexion_pulgar > UMBRAL_FLEX_PULGAR or 
            cambio_rotacion_pulgar > 0): # La rotación ya fue filtrada por su UMBRAL_ROTACION arriba
            
            if MANO_CONECTADA and mano is not None:
                mano.send_raw(vector_modbus_actual.tolist())
            
            ultimo_vector_enviado = vector_modbus_actual.copy()
            ultimo_envio = time.time()
```
- **Control de Frecuencia** (`FRECUENCIA_ENVIO = 0.05`): Limita las tramas TCP/IP a un máximo de $20\,\text{Hz}$ ($50\,\text{ms}$ entre paquetes) para no saturar el bus Modbus del controlador de la mano.
- **Evaluación Desacoplada de Deltas**: En lugar de aplicar un único umbral global que ahogaría los movimientos finos del pulgar, se evalúan tres condiciones en paralelo sobre el último vector enviado (`ultimo_vector_enviado`):
1. `cambio_dedos_principales > 100`: Verifica si alguno de los 4 dedos largos (`0:4`) superó las 100 unidades.
2. `cambio_flexion_pulgar > 15`: Permite que la flexión del pulgar (`pos(4)`) dispare una actualización con variaciones sutiles mayores a 15 unidades.
3. `cambio_rotacion_pulgar > 0`: Como `vector_modbus_actual[5]` ya pasó por su propio filtro de 50 unidades (`UMBRAL_ROTACION`), cualquier cambio mayor a `0` indica un giro válido que debe enviarse de inmediato. 

## Bloque 10: Renderizado Visual (HUD), Menú de Rutinas y Cierre Seguro
```bash
# --- RENDERIZADO VISUAL ---
cv2.putText(color_image, "'e' Toggle Tracking | 's' Guardar | 'r' Menu Rutinas | 'q' Salir", (10, 30), 
            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
```
- **HUD de Telemetría en Tiempo Real**:
1. Muestra siempre en pantalla los valores calculados actuales de `Flex(4)` y `Rot(5)` junto con el diagnóstico exacto del estado de rotación: `[ACTIVA]`, `[Oclusion Indice -> FORZADO 1000]` o `[Flexion Pulgar -> BLOQUEADA]`.
2. El dibujo de textos e `imshow` se ejecuta fuera de los condicionales de detección de mano, garantizando que el video nunca se congele si la mano sale temporalmente del encuadre.
- **Interacción por Teclado** (`cv2.waitKey(1)`):
1. `'e'` **(Toggle Tracking)**: Activa o pausa el envío de tramas Modbus en tiempo real y reinicia `ultimo_vector_enviado = np.array(ABIERTO)` para forzar una actualización inmediata al reanudar.
2. `'s'` **(Guardar Gesto)**: Pausa automáticamente la teleoperación y solicita en consola un nombre para almacenar el `vector_modbus_actual` en `gestos_guardados_3d.csv`.
3. `'r'` **(Menú de Rutinas 3D)**: Pausa la teleoperación y despliega en consola el submenú para: (1) ejecutar un gesto individual guardado, (2) concatenar una nueva secuencia de gestos por índices, guardarla y ejecutarla, o (3) importar y reproducir una rutina existente desde `rutinas_guardadas_3d.csv` con pausas de $1.0\,\text{s}$ entre posturas.
4. `'q'` **(Salir) y Bloque** `finally`: Rompe el bucle principal y ejecuta de forma garantizada el bloque `finally:`, el cual detiene el flujo de la cámara RealSense (`pipeline.stop()`), destruye las ventanas de OpenCV, envía la consigna `ABIERTO` (`[1000, 1000, 1000, 1000, 1000, 1000]`) para dejar la mano robótica en posición de reposo seguro y cierra el socket TCP (`mano.close()`).  

## Ejecución del Programa

In [7]:
if __name__ == "__main__":
    main()

Conectando a la mano en 192.168.123.211 ...
Configurando fuerza segura (300|500(4)) en todos los motores...

 BIENVENIDO AL SISTEMA DE VISIÓN 3D
1. Iniciar cámara de ejecución (Usar calibración predeterminada)
2. Calibrar mi mano en 4 fases (Recomendado para mayor precisión)


INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1792765088.735221   15668 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1792765088.758523   15668 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Seleccione una opción (1 o 2):  1



=== CÁMARA DE EJECUCIÓN CONTINUA ===
 'e' : Iniciar / Pausar Seguimiento en Tiempo Real (Toggle)
 's' : Guardar gesto actual en CSV
 'r' : Abrir Menú de Rutinas (Consola)
 'q' : Salir

>> MODO TELEOPERACIÓN: ACTIVADO
Cámara apagada y recursos liberados.
